# 02b SQL Business Queries - Standardized Input

## Purpose

This notebook is the standardized-input version of `02_sql_business_queries.ipynb`. It reproduces the same five SQL business analyses using the cleaned standardized sales data created by `01b_data_cleaning_standardized_input.ipynb`.

This is a preparation layer for future reusable pipeline refactoring. It does not replace the original workflow or change the validated project metrics.

## Input and output isolation

The notebook reads only the standardized cleaned sales input:

```text
data/processed_standardized/clean_sales.csv
```

It writes the five business-query CSVs only to `outputs_standardized/`. The original `data/processed/`, `outputs/`, and SQLite database files are not modified. An in-memory SQLite database is used for the analysis.

`stock_code` is the SKU key. `description` is retained only as a deterministic display field so description variations do not split one SKU into multiple summary rows.

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

# Resolve paths whether the notebook runs from the project root or notebooks/.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
clean_sales_path = project_root / "data" / "processed_standardized" / "clean_sales.csv"
outputs_dir = project_root / "outputs_standardized"

if not clean_sales_path.exists():
    raise FileNotFoundError(
        "Standardized cleaned sales input not found. Run "
        "notebooks/01b_data_cleaning_standardized_input.ipynb first. "
        f"Expected file: {clean_sales_path}"
    )

sales = pd.read_csv(
    clean_sales_path,
    dtype={
        "invoice_no": "string",
        "stock_code": "string",
        "description": "string",
        "customer_id": "string",
        "country": "string",
    },
)

required_columns = {
    "invoice_no", "stock_code", "description", "quantity",
    "unit_price", "country", "invoice_month", "revenue",
}
missing_columns = sorted(required_columns - set(sales.columns))
if missing_columns:
    raise ValueError("Missing columns in standardized clean sales: " + ", ".join(missing_columns))

outputs_dir.mkdir(parents=True, exist_ok=True)

# Keep the SQL work transient so no database file is created or overwritten.
conn = sqlite3.connect(":memory:")
sales.to_sql("clean_sales", conn, if_exists="replace", index=False)

print("Standardized clean product sales rows loaded:", len(sales))
print("Input file:", clean_sales_path)
print("Output directory:", outputs_dir)

## Database and data-scope validation

These checks confirm that the standardized cleaned data is available in SQLite and that known non-product stock codes remain excluded from the business analysis.

In [ ]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn,
)

row_count = pd.read_sql(
    "SELECT COUNT(*) AS row_count FROM clean_sales;",
    conn,
)

non_product_check_query = """
SELECT
    stock_code,
    COUNT(*) AS row_count,
    ROUND(SUM(revenue), 2) AS total_revenue
FROM clean_sales
WHERE stock_code IN ('POST', 'DOT', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'D', 'M')
GROUP BY stock_code;
"""

non_product_check = pd.read_sql(non_product_check_query, conn)

display(tables)
display(row_count)
display(non_product_check)
print("Non-product stock codes remaining in clean_sales:", len(non_product_check))

## Top SKU revenue contribution

This query identifies the 20 SKUs with the highest revenue contribution. It preserves notebook 02's revenue, unit, order-count, and average-price measures while aggregating at the `stock_code` key.

In [ ]:
top_sku_query = """
SELECT
    stock_code,
    MIN(description) AS description,
    SUM(quantity) AS total_units,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS order_count,
    ROUND(AVG(unit_price), 2) AS avg_unit_price
FROM clean_sales
GROUP BY stock_code
ORDER BY total_revenue DESC
LIMIT 20;
"""

top_sku = pd.read_sql(top_sku_query, conn)
top_sku.to_csv(outputs_dir / "top_sku_revenue_contribution.csv", index=False)
top_sku

## Top SKU unit contribution

This query identifies the 20 highest-volume SKUs. High-volume products may need stable replenishment even when their unit prices are relatively low.

In [ ]:
top_units_query = """
SELECT
    stock_code,
    MIN(description) AS description,
    SUM(quantity) AS total_units,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS order_count,
    ROUND(AVG(unit_price), 2) AS avg_unit_price
FROM clean_sales
GROUP BY stock_code
ORDER BY total_units DESC
LIMIT 20;
"""

top_units = pd.read_sql(top_units_query, conn)
top_units.to_csv(outputs_dir / "top_sku_unit_contribution.csv", index=False)
top_units

## Long-tail SKU identification

As in notebook 02, a long-tail SKU has total units of 20 or fewer. The result is ordered by units and then revenue, with a maximum of 50 rows.

In [ ]:
long_tail_query = """
SELECT
    stock_code,
    MIN(description) AS description,
    SUM(quantity) AS total_units,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS order_count,
    COUNT(DISTINCT invoice_month) AS active_months,
    ROUND(AVG(unit_price), 2) AS avg_unit_price
FROM clean_sales
GROUP BY stock_code
HAVING SUM(quantity) <= 20
ORDER BY total_units ASC, total_revenue ASC
LIMIT 50;
"""

long_tail_skus = pd.read_sql(long_tail_query, conn)
long_tail_skus.to_csv(outputs_dir / "long_tail_skus.csv", index=False)
long_tail_skus

## Country-level demand summary

This query summarizes units, revenue, orders, and distinct SKUs by country to support later warehouse allocation and fulfillment analysis.

In [ ]:
country_demand_query = """
SELECT
    country,
    SUM(quantity) AS total_units,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS order_count,
    COUNT(DISTINCT stock_code) AS unique_skus
FROM clean_sales
GROUP BY country
ORDER BY total_revenue DESC;
"""

country_demand = pd.read_sql(country_demand_query, conn)
country_demand.to_csv(outputs_dir / "country_demand_summary.csv", index=False)
country_demand.head(20)

## Monthly sales trend

This query summarizes monthly unit volume, revenue, order count, and distinct SKUs using the same logic as notebook 02.

In [ ]:
monthly_sales_query = """
SELECT
    invoice_month,
    SUM(quantity) AS total_units,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS order_count,
    COUNT(DISTINCT stock_code) AS unique_skus
FROM clean_sales
GROUP BY invoice_month
ORDER BY invoice_month;
"""

monthly_sales = pd.read_sql(monthly_sales_query, conn)
monthly_sales.to_csv(outputs_dir / "monthly_sales_trend.csv", index=False)
monthly_sales

## Output summary

The final cell lists each standardized output and its DataFrame shape. These parallel outputs are intended for future reusable pipeline work and do not replace the original workflow outputs.

In [ ]:
output_summary = {
    "outputs_standardized/top_sku_revenue_contribution.csv": top_sku.shape,
    "outputs_standardized/top_sku_unit_contribution.csv": top_units.shape,
    "outputs_standardized/long_tail_skus.csv": long_tail_skus.shape,
    "outputs_standardized/country_demand_summary.csv": country_demand.shape,
    "outputs_standardized/monthly_sales_trend.csv": monthly_sales.shape,
}

print("===== 02b Standardized-Input SQL Business Queries Summary =====")
print("Standardized clean product sales rows loaded:", len(sales))
print("Non-product stock codes remaining in clean_sales:", len(non_product_check))
print("\nOutput files and shapes:")
for output_file, shape in output_summary.items():
    print(f"- {output_file}: {shape}")

conn.close()